# Additional Test — Small Chunk Size: 1 Patch per Chunk

## Operation Measured
Read **1,000 contiguous patches** (adjacent patch_id) from a **1M-patch Zarr array** into memory, BUT set the chunk size to (1, 32, 32, 3) — one patch per chunk.

This tests the impact of very small (worst-case) chunking for batch reading. Benchmark configuration and timing methodology match the main test above.

In [ ]:
import os
import shutil
import numpy as np
import zarr
import time

NUM_PATCHES = 1_000_000
PATCH_SHAPE = (32, 32, 3)
DTYPE = 'uint8'
READ_COUNT = 1000
START_ID = 500_000
ZARR_PATH_SM = '/tmp/benchmark_contiguous_1000_chunksz1.zarr'
CHUNK_SIZE_SM = 1  # 1 patch per chunk

try:
    # --- SETUP (not timed) ---
    if os.path.exists(ZARR_PATH_SM):
        shutil.rmtree(ZARR_PATH_SM)

    print('Creating Zarr array (1 patch/chunk, shape=(1M, 32, 32, 3))...')
    z_sm = zarr.open(
        ZARR_PATH_SM,
        mode='w',
        shape=(NUM_PATCHES,) + PATCH_SHAPE,
        chunks=(CHUNK_SIZE_SM,) + PATCH_SHAPE,
        dtype=DTYPE,
    )

    # Seed with random uint8 data in batches of 10k
    print('Seeding data...')
    rng = np.random.default_rng(2024)
    seed_batch = 10_000
    for i in range(0, NUM_PATCHES, seed_batch):
        end = min(i + seed_batch, NUM_PATCHES)
        z_sm[i:end] = rng.integers(0, 256, size=(end - i,) + PATCH_SHAPE, dtype=np.uint8)
    print('Seeding complete.')

    # Warm-up: small read before timing
    _ = z_sm[0:1]

    # --- TIMED BLOCK (single trial) ---
    print(f'Reading {READ_COUNT} contiguous patches starting at patch_id={START_ID}...')
    t0 = time.perf_counter()
    batch = z_sm[START_ID : START_ID + READ_COUNT]
    t1 = time.perf_counter()

    elapsed = t1 - t0
    throughput = READ_COUNT / elapsed
    print(f'RESULT: Elapsed={elapsed:.6f}s, Throughput={throughput:,.0f} patches/s')
    print(f'Batch shape: {batch.shape}')
    print(f'Batch dtype: {batch.dtype}')

    # --- 5-TRIAL ROBUSTNESS ---
    print('\nRunning 5 trials for robustness...')
    times = []
    for trial in range(5):
        _ = z_sm[0:1]  # minor warm-up per trial
        t0 = time.perf_counter()
        batch = z_sm[START_ID : START_ID + READ_COUNT]
        t1 = time.perf_counter()
        times.append(t1 - t0)
        print(f'  Trial {trial+1}: {times[-1]:.6f}s')

    median_elapsed = float(np.median(times))
    median_throughput = READ_COUNT / median_elapsed
    print(f'\nMedian over 5 trials: {median_elapsed:.6f}s, {median_throughput:,.0f} patches/s')

except Exception as e:
    print(f'ERROR: {e}')
    import traceback
    traceback.print_exc()
    raise
finally:
    if os.path.exists(ZARR_PATH_SM):
        shutil.rmtree(ZARR_PATH_SM)
        print('\nZarr array cleaned up.')

/home/ray/anaconda3/lib/python3.13/site-packages/google_crc32c/__config__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/ray/anaconda3/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


Creating Zarr array (1 patch/chunk, shape=(1M, 32, 32, 3))...
Seeding data...


# Result Summary — 1 Patch per Chunk

(Fill results after execution.)

| Metric            | Single Read       | Median (5 trials) |
| ----------------- | ----------------- | ----------------- |
| Elapsed time      | ... s             | ... s            |
| Throughput        | ... patches/s     | ... patches/s     |
| Batch shape       | (1000, 32, 32, 3) | —                 |
| dtype             | uint8             | —                 |

Suggested CSV result for this row:

```
"...s, ~... patches/s (chunk=1)"
```